# Week 7 — From a protein table to biology: PR1 heart proteomics

**Adapted working copy of `02_downstream_analysis.ipynb` for the PR1 human heart-valve dataset**

This notebook follows the course workflow (filter → log-transform → inspect missing values → check normalization → PCA → differential testing), but replaces the Week 7 practice datasets with your PR1 data.

**Primary comparison:** aortic stenosis (AS) vs healthy donor controls (CON).

**Secondary group:** mitral regurgitation (MR). We keep its metadata for later exploratory comparisons, but the first differential-abundance analysis below focuses on AS vs CON.

> Important: this is a working analysis template. Inspect the plots and assumptions before interpreting results. It does not establish biological conclusions by itself.

## A0 · Load the PR1 data

In Google Colab, upload **both** files:
- `PR1_heart_valve_disease_proteinGroups.csv`
- `PR1_heart_valve_disease_samples.csv`

The samples file tells us which LFQ column belongs to which disease group and sex.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA
from google.colab import files

uploaded = files.upload()

protein_file = next((name for name in uploaded if "proteinGroups" in name), None)
samples_file = next((name for name in uploaded if "samples" in name), None)

if protein_file is None or samples_file is None:
    raise ValueError(
        "Please upload both PR1_heart_valve_disease_proteinGroups.csv "
        "and PR1_heart_valve_disease_samples.csv"
    )

pg = pd.read_csv(protein_file, low_memory=False)
samples = pd.read_csv(samples_file)

print("Protein table shape:", pg.shape)
print("Samples table shape:", samples.shape)
display(samples.head())
display(pg.head(3))

## A0.1 · Check the sample metadata and match LFQ columns

We use the `samples` table rather than guessing group membership from column names. This protects us against comparing the wrong samples.

In [ ]:
required_sample_columns = {"column", "sample", "group", "sex"}
missing_metadata_columns = required_sample_columns - set(samples.columns)
if missing_metadata_columns:
    raise ValueError(f"Missing columns in samples file: {missing_metadata_columns}")

lfq = [c for c in pg.columns if c.startswith("LFQ intensity ")]
sample_columns = samples["column"].astype(str).tolist()

missing_in_protein_table = sorted(set(sample_columns) - set(pg.columns))
if missing_in_protein_table:
    raise ValueError(f"Sample columns missing from protein table: {missing_in_protein_table[:5]}")

if samples["column"].duplicated().any():
    raise ValueError("The samples file contains duplicate LFQ column names.")

# Keep the LFQ columns in the exact order listed in the samples metadata.
lfq = sample_columns

print(f"Matched {len(lfq)} LFQ columns to {len(samples)} sample records.")
print("Groups:")
print(samples.groupby(["group", "sex"]).size().unstack(fill_value=0))

## A1 · Filter protein identifications

The PR1 README says the original table has **not** been filtered. Its flags are named `Reverse`, `Potential contaminant`, and `Only identified by site` (note that the PR1 column is called `Potential contaminant`, not `Contaminant` as in the course benchmark).

For the main analysis, we exclude rows marked `+` in **any** of these three columns.

In [ ]:
flag_columns = ["Reverse", "Potential contaminant", "Only identified by site"]
missing_flag_columns = set(flag_columns) - set(pg.columns)
if missing_flag_columns:
    raise ValueError(f"Expected identification flag columns are missing: {missing_flag_columns}")

flagged = pd.DataFrame({
    col: pg[col].fillna("").astype(str).str.strip().eq("+")
    for col in flag_columns
}, index=pg.index)

print("Rows with each flag (counts can overlap):")
print(flagged.sum())

remove_any_flag = flagged.any(axis=1)
d = pg.loc[~remove_any_flag].copy()

print(f"Original protein groups: {len(pg):,}")
print(f"Removed because at least one flag was present: {int(remove_any_flag.sum()):,}")
print(f"Remaining protein groups: {len(d):,}")

## A2 · Convert zero values to missing values and log-transform

In this dataset, an LFQ value of `0` means **not quantified in that sample**. It is not a measured abundance of zero. We therefore replace zero with `NaN` before taking the log2 transform.

In [ ]:
# Convert LFQ columns to numeric, then mark non-positive values as missing.
d[lfq] = d[lfq].apply(pd.to_numeric, errors="coerce")
zero_count = int((d[lfq] == 0).sum().sum())
print(f"Zero LFQ measurements before replacement: {zero_count:,}")

X = np.log2(d[lfq].mask(d[lfq] <= 0))

print("Log2-transformed matrix shape:", X.shape)
print("Missing LFQ measurements after zero replacement:", int(X.isna().sum().sum()))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
raw_values = d[lfq].stack().dropna()
log_values = X.stack().dropna()
ax[0].hist(raw_values, bins=60)
ax[0].set(title="Raw LFQ intensities", xlabel="LFQ intensity", ylabel="Count")
ax[1].hist(log_values, bins=60)
ax[1].set(title="Log2 LFQ intensities", xlabel="log2 intensity", ylabel="Count")
plt.tight_layout()
plt.show()

## A3 · Inspect missing values

First, count missing measurements in each sample. Then examine whether missingness is more common among proteins with lower average measured abundance. This is a diagnostic: it does not prove why a value is missing.

In [ ]:
missing_per_sample = X.isna().sum().sort_values(ascending=False)
print("Missing values per sample:")
display(missing_per_sample.to_frame("missing_LFQ_values"))

mean_log_intensity = X.mean(axis=1, skipna=True)
fraction_missing = X.isna().mean(axis=1)
missingness_check = pd.DataFrame({
    "mean_log2_intensity": mean_log_intensity,
    "fraction_missing": fraction_missing
}).dropna()

# Divide proteins into abundance deciles; duplicates='drop' handles tied values safely.
missingness_check["abundance_decile"] = pd.qcut(
    missingness_check["mean_log2_intensity"], q=10, labels=False, duplicates="drop"
)
missing_by_decile = missingness_check.groupby("abundance_decile", observed=True)["fraction_missing"].mean()

ax = missing_by_decile.plot(kind="bar", figsize=(7, 4))
ax.set(xlabel="Mean log2 abundance decile (low to high)",
       ylabel="Mean fraction missing",
       title="Missingness versus measured abundance")
plt.tight_layout()
plt.show()

## A3.1 · Define the primary comparison and filter by detection frequency

The main analysis compares **aortic stenosis** with **control (donor heart)**. We use the metadata table to select these samples.

For this first pass, retain a protein if it has a measured value in at least **50% of samples in each of the two groups**. With the current group sizes, that means at least 21 of 41 AS samples and 9 of 17 controls. This is an analysis choice, not a universal rule; later, check whether conclusions are robust to a stricter threshold.

In [ ]:
AS_GROUP = "aortic stenosis"
CONTROL_GROUP = "control (donor heart)"

as_cols = samples.loc[samples["group"].eq(AS_GROUP), "column"].tolist()
control_cols = samples.loc[samples["group"].eq(CONTROL_GROUP), "column"].tolist()

if not as_cols or not control_cols:
    raise ValueError("Could not find the expected aortic stenosis and control groups. Check samples['group'].")

min_as = int(np.ceil(0.50 * len(as_cols)))
min_control = int(np.ceil(0.50 * len(control_cols)))

valid = (
    (X[as_cols].notna().sum(axis=1) >= min_as) &
    (X[control_cols].notna().sum(axis=1) >= min_control)
)

X_primary = X.loc[valid, as_cols + control_cols].copy()
d_primary = d.loc[valid].copy()

print(f"Aortic stenosis samples: {len(as_cols)}; minimum detected per protein: {min_as}")
print(f"Control samples: {len(control_cols)}; minimum detected per protein: {min_control}")
print(f"Protein groups before detection filter: {len(X):,}")
print(f"Protein groups after detection filter: {len(X_primary):,}")
print(f"Missing values remaining in primary comparison: {int(X_primary.isna().sum().sum()):,}")

## A4 · Check sample distributions before deciding about normalization

The course benchmark demonstrates why median normalization can be misleading when a large fraction of proteins truly shifts between conditions. PR1 LFQ values come from MaxQuant's LFQ workflow, so **do not automatically repeat median normalization**. First inspect sample distributions and sample medians. Only apply additional normalization if you have evidence and can justify it.

In [ ]:
primary_samples = samples[samples["column"].isin(as_cols + control_cols)].copy()
primary_samples["column"] = pd.Categorical(
    primary_samples["column"], categories=as_cols + control_cols, ordered=True
)
primary_samples = primary_samples.sort_values("column")
plot_cols = primary_samples["column"].astype(str).tolist()

fig, ax = plt.subplots(figsize=(15, 5))
ax.boxplot([X_primary[c].dropna() for c in plot_cols], showfliers=False)
ax.set_xticks(range(1, len(plot_cols) + 1))
ax.set_xticklabels([c.replace("LFQ intensity ", "") for c in plot_cols], rotation=90)
ax.set_ylabel("log2 LFQ intensity")
ax.set_title("Log2 intensity distributions by sample (AS and controls)")
plt.tight_layout()
plt.show()

sample_medians = X_primary.median(axis=0)
print("Sample median log2 intensities:")
display(sample_medians.to_frame("median_log2_LFQ"))

## A5 · Prepare a complete matrix for PCA

PCA requires a complete numerical matrix. For this **exploratory PCA only**, the code below fills missing values for each protein with that protein's median across the AS and control samples.

This is a pragmatic visualization choice, not the differential-abundance test. Imputation can affect PCA, especially when missingness differs between groups, so interpret the plot cautiously. The statistical test below uses the log2 values with missing values left as missing.

In [ ]:
# Transpose so rows are samples and columns are proteins.
# The row-wise protein medians in X_primary become the corresponding column fill values.
protein_medians = X_primary.median(axis=1)
X_pca = X_primary.T.fillna(protein_medians)

if X_pca.isna().any().any():
    raise ValueError("Some proteins have no observed values and could not be imputed for PCA.")

print("PCA matrix (samples x proteins):", X_pca.shape)

## A6 · PCA: do the AS and control samples look different?

Each point is one biopsy sample. Colour represents disease group and marker shape represents sex. PCA is exploratory: visible separation is not proof of a statistically significant difference, and overlap does not mean there are no protein-level changes.

In [ ]:
pca = PCA(n_components=2)
pc_scores = pca.fit_transform(X_pca.values)
explained = pca.explained_variance_ratio_

pca_meta = samples.set_index("column").loc[X_pca.index].copy()
colors = {AS_GROUP: "tab:orange", CONTROL_GROUP: "tab:blue"}
markers = {"female": "^", "male": "o"}

fig, ax = plt.subplots(figsize=(8, 6))
for group in [AS_GROUP, CONTROL_GROUP]:
    for sex in ["female", "male"]:
        mask = (pca_meta["group"].eq(group) & pca_meta["sex"].eq(sex)).to_numpy()
        if mask.any():
            ax.scatter(
                pc_scores[mask, 0], pc_scores[mask, 1],
                label=f"{group} – {sex}",
                color=colors[group], marker=markers[sex],
                s=75, alpha=0.85, edgecolor="black", linewidth=0.4
            )

ax.set_xlabel(f"PC1 ({explained[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({explained[1]:.1%} variance)")
ax.set_title("PCA of AS and control heart proteomes")
ax.axhline(0, color="lightgrey", linewidth=0.7)
ax.axvline(0, color="lightgrey", linewidth=0.7)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Sample-to-sample Pearson correlation based on the same PCA input matrix.
corr = X_pca.T.corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, vmin=0, vmax=1, cmap="viridis")
ax.set_title("Sample correlation (AS and controls)")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels([s.replace("LFQ intensity ", "") for s in corr.columns], rotation=90, fontsize=7)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels([s.replace("LFQ intensity ", "") for s in corr.index], fontsize=7)
plt.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

## A7 · Test each protein: aortic stenosis vs controls

We use a two-sided **Welch t-test** on log2 LFQ values. Welch's test does not assume equal variances. Missing values are omitted for each protein; they are **not imputed for this test**. We then apply Benjamini–Hochberg false-discovery-rate correction across the tested protein groups.

A positive log2 fold change means higher average abundance in AS; a negative value means lower average abundance in AS.

In [ ]:
t = stats.ttest_ind(
    X_primary[as_cols].to_numpy(),
    X_primary[control_cols].to_numpy(),
    axis=1,
    equal_var=False,
    nan_policy="omit"
)

mean_as = X_primary[as_cols].mean(axis=1, skipna=True)
mean_control = X_primary[control_cols].mean(axis=1, skipna=True)

res = pd.DataFrame({
    "Protein IDs": d_primary["Protein IDs"].astype(str).values,
    "Gene names": d_primary["Gene names"].fillna("").astype(str).values,
    "n_AS": X_primary[as_cols].notna().sum(axis=1).values,
    "n_Control": X_primary[control_cols].notna().sum(axis=1).values,
    "mean_log2_AS": mean_as.values,
    "mean_log2_Control": mean_control.values,
    "log2FC_AS_vs_Control": (mean_as - mean_control).values,
    "p_value": np.asarray(t.pvalue)
}, index=X_primary.index)

res["q_value"] = np.nan
valid_p = res["p_value"].notna()
res.loc[valid_p, "q_value"] = stats.false_discovery_control(
    res.loc[valid_p, "p_value"].to_numpy(), method="bh"
)
res["significant_q05"] = res["q_value"] < 0.05

print("Proteins tested:", len(res))
print("Significant at BH-adjusted q < 0.05:", int(res["significant_q05"].sum()))
print("Higher in AS:", int((res["significant_q05"] & (res["log2FC_AS_vs_Control"] > 0)).sum()))
print("Lower in AS:", int((res["significant_q05"] & (res["log2FC_AS_vs_Control"] < 0)).sum()))

display(res.sort_values("q_value").head(20))

## A7.1 · Volcano plot

The x-axis is the log2 fold change (AS vs controls). The y-axis is the negative log10 of the raw p-value. Colour indicates whether the protein passes the adjusted q < 0.05 threshold. Consider both statistical significance and effect size when interpreting proteins.

In [ ]:
plot_res = res.copy()
plot_res["minus_log10_p"] = -np.log10(plot_res["p_value"].clip(lower=np.finfo(float).tiny))
plot_res["status"] = np.where(
    plot_res["q_value"] < 0.05,
    np.where(plot_res["log2FC_AS_vs_Control"] > 0, "Higher in AS", "Lower in AS"),
    "Not significant"
)

palette = {"Higher in AS": "tab:red", "Lower in AS": "tab:blue", "Not significant": "lightgrey"}
fig, ax = plt.subplots(figsize=(9, 6))
for status, color in palette.items():
    subset = plot_res[plot_res["status"] == status]
    ax.scatter(subset["log2FC_AS_vs_Control"], subset["minus_log10_p"],
               s=12, alpha=0.65, c=color, label=status)
ax.axhline(-np.log10(0.05), color="grey", linestyle="--", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("log2 fold change (aortic stenosis / control)")
ax.set_ylabel("−log10(raw p-value)")
ax.set_title("Differential protein abundance: AS vs controls")
ax.legend()
plt.tight_layout()
plt.show()

## A8 · Save the cleaned data and statistical results

These files can be used later for enrichment analysis and Cytoscape. Keep the filtered log2 matrix (with missing values still marked as `NaN`) separate from the results table.

In [ ]:
# Save the filtered protein table and log2 abundance matrix.
cleaned_proteins = d.loc[X.index].copy()
cleaned_proteins.to_csv("PR1_filtered_proteinGroups.csv", index=False)
X.to_csv("PR1_log2_LFQ_matrix_with_NaN.csv")
res.sort_values("q_value").to_csv("PR1_AS_vs_Control_differential_results.csv", index=False)

print("Saved:")
print("- PR1_filtered_proteinGroups.csv")
print("- PR1_log2_LFQ_matrix_with_NaN.csv")
print("- PR1_AS_vs_Control_differential_results.csv")

## What is intentionally not copied from the original exercise?

- The original Part A has a **HeLa/E. coli benchmark** with six L/H samples. Its `Species`, `L`, and `H` logic does not apply to PR1.
- The original Part A “check against the truth” section is specific to that benchmark and must be skipped.
- The original Part B is a **FLCN knockout kidney-cell** study. Its sample names, groups, and biological interpretation do not apply to heart tissue.
- The original enrichment code can be adapted later, using your PR1 gene list and the proteins actually measured in PR1 as the background.

**Next analysis after this notebook section:** test the disease-by-sex interaction with an appropriate model, then perform GO/pathway enrichment and build a human protein-interaction network for Cytoscape. Do not treat the PCA or volcano plot alone as a biological conclusion.